# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Tanzina-Aranya-Islam/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

### Signal 1 — Impressions / Volume

I tested whether higher search volume is associated with more clicks.

The bucket table shows a clear increase in mean clicks as impressions increase:
from 0.0078 clicks for 1–5 impressions to 2.9420 clicks for 501+ impressions.

**Verdict: CONFIRMED**

This is an observed association in the March data, not evidence of causation.

### Signal 2 — Position vs CTR

I tested whether pages with better average position tend to have higher CTR.

The bucket table shows a consistent decrease in mean CTR as position gets worse:
from 0.004756 for positions 1–3 to 0.000494 for positions 51+.

**Verdict: CONFIRMED**

This supports the directional assumption behind a CTR-vs-position review, but it does not by itself prove that a page needs a CTR fix.

In [8]:
import pandas as pd

df_march = pd.read_parquet(
    "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet"
)

print("Rows:", len(df_march))
print("Columns:", df_march.columns.tolist())

Rows: 9841378
Columns: ['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']


In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import numpy as np
import pandas as pd


audit = df_march[
    [
        "gsc_impressions",
        "gsc_clicks",
        "gsc_avg_position"
    ]
].copy()


audit["ctr"] = np.where(
    audit["gsc_impressions"] > 0,
    audit["gsc_clicks"] / audit["gsc_impressions"],
    np.nan
)

# Signal 1: Impressions / Volume
audit["impression_bucket"] = pd.cut(
    audit["gsc_impressions"],
    bins=[-1, 0, 5, 20, 100, 500, np.inf],
    labels=["0", "1-5", "6-20", "21-100", "101-500", "501+"]
)

volume_table = (
    audit.groupby("impression_bucket", observed=False)
    .agg(
        n=("gsc_impressions", "size"),
        mean_clicks=("gsc_clicks", "mean"),
        mean_ctr=("ctr", "mean")
    )
    .reset_index()
)

print("SIGNAL 1 — IMPRESSIONS / VOLUME")
display(volume_table)



# Signal 2: Position vs CTR
audit["position_bucket"] = pd.cut(
    audit["gsc_avg_position"],
    bins=[-np.inf, 3, 10, 20, 50, np.inf],
    labels=["1-3", "4-10", "11-20", "21-50", "51+"]
)

position_table = (
    audit.groupby("position_bucket", observed=False)
    .agg(
        n=("gsc_avg_position", "size"),
        mean_ctr=("ctr", "mean"),
        median_ctr=("ctr", "median")
    )
    .reset_index()
)

print("\nSIGNAL 2 — POSITION vs CTR")
display(position_table)

SIGNAL 1 — IMPRESSIONS / VOLUME


,impression_bucket,n,mean_clicks,mean_ctr
0,0,6230317,0.000000,NaN
1,1-5,1100192,0.007780,0.003864
2,6-20,891854,0.027340,0.002369
3,21-100,985532,0.143100,0.002869
4,101-500,532347,0.658058,0.003100
5,501+,101136,2.942019,0.002803



SIGNAL 2 — POSITION vs CTR


,position_bucket,n,mean_ctr,median_ctr
0,1-3,727362,0.004756,0.0
1,4-10,1456122,0.003473,0.0
2,11-20,519223,0.002770,0.0
3,21-50,631491,0.001638,0.0
4,51+,276863,0.000494,0.0


## 2. Build the ranked queue (writes the CSV)

I rank pages using a simple screening score based on impressions and clicks.

**Score = impressions / (clicks + 1)**

Pages with high impressions and very few clicks receive a higher score and are
assigned the action **REVIEW_PAGE**.

### Reason code

- **HIGH_VOLUME_LOW_CLICKS** — the page has high impressions relative to clicks.
- **NO_IMPRESSIONS** — the page has no impressions, so there is no search-volume signal to review.

This is a baseline prioritization rule, not a prediction of SEO performance.
It uses only observed March 2026 data and does not use future labels or existing
decision outputs.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import numpy as np
import pandas as pd
from pathlib import Path

queue = df_march[
    [
        "client_hash_id",
        "content_hash_id",
        "report_date",
        "gsc_impressions",
        "gsc_clicks"
    ]
].copy()


queue["ctr"] = np.where(
    queue["gsc_impressions"] > 0,
    queue["gsc_clicks"] / queue["gsc_impressions"],
    np.nan
)


queue["score"] = np.where(
    queue["gsc_impressions"] > 0,
    queue["gsc_impressions"] / (queue["gsc_clicks"] + 1),
    0
)


queue["action"] = np.where(
    queue["score"] > 0,
    "REVIEW_PAGE",
    "NO_ACTION"
)


queue["reason_code"] = np.where(
    queue["score"] > 0,
    "HIGH_VOLUME_LOW_CLICKS",
    "NO_IMPRESSIONS"
)


output = queue[
    [
        "client_hash_id",
        "content_hash_id",
        "report_date",
        "score",
        "action",
        "reason_code"
    ]
].copy()


top20 = output.nlargest(20, "score").copy()
top20.insert(0, "rank", range(1, len(top20) + 1))

output_path = Path("work/outputs/baseline_action_score.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

output.to_csv(output_path, index=False)

print("CSV saved:", output_path)
print("Total rows:", len(output))
print("\nTop 20:")
display(top20)

CSV saved: work/outputs/baseline_action_score.csv
Total rows: 9841378

Top 20:


,rank,client_hash_id,content_hash_id,report_date,score,action,reason_code
103661,1,client_62f4a7e64f5e0096,content_945d6ff91386c817,2026-03-04,37368.0,REVIEW_PAGE,HIGH_VOLUME_LOW_CLICKS
8882581,2,client_73cda7b4e4f265ea,content_fec55986a1868d62,2026-03-30,33383.0,REVIEW_PAGE,HIGH_VOLUME_LOW_CLICKS
8550024,3,client_23a62021009f63c4,content_44f34c0a90047651,2026-03-27,32958.0,REVIEW_PAGE,HIGH_VOLUME_LOW_CLICKS
9451315,4,client_73cda7b4e4f265ea,content_fec55986a1868d62,2026-03-31,31472.0,REVIEW_PAGE,HIGH_VOLUME_LOW_CLICKS
766551,5,client_73cda7b4e4f265ea,content_9c057b66c30a3abb,2026-03-02,28973.0,REVIEW_PAGE,HIGH_VOLUME_LOW_CLICKS
599344,6,client_73cda7b4e4f265ea,content_9c057b66c30a3abb,2026-03-01,28947.0,REVIEW_PAGE,HIGH_VOLUME_LOW_CLICKS
6438283,7,client_62f4a7e64f5e0096,content_34a70fea29d15f24,2026-03-22,27410.0,REVIEW_PAGE,HIGH_VOLUME_LOW_CLICKS
803453,8,client_73cda7b4e4f265ea,content_9c057b66c30a3abb,2026-03-03,24233.0,REVIEW_PAGE,HIGH_VOLUME_LOW_CLICKS
9655081,9,client_23a62021009f63c4,content_44f34c0a90047651,2026-03-28,20042.0,REVIEW_PAGE,HIGH_VOLUME_LOW_CLICKS
4062723,10,client_e547b89c05043229,content_757b1fa67827358d,2026-03-13,19301.0,REVIEW_PAGE,HIGH_VOLUME_LOW_CLICKS


## 3. Top-20 review

The ranked queue prioritizes pages with high impressions and very low clicks.
This is useful for finding pages that may deserve CTR review, but the score is
only a screening heuristic. A high score does not prove that a page has a
content problem or needs an SEO change.

### Review notes

1. **Rank 1 — REVIEW_PAGE:** Very high impressions with 0 clicks, so it receives a high score. It could be wrong if the page/query mix legitimately produces little click activity or if the source data is incomplete.

2. **Rank 2 — REVIEW_PAGE:** High impressions and 0 clicks make this a strong review candidate. It could be wrong if the page is exposed for queries where clicks are not expected.

3. **Rank 3 — REVIEW_PAGE:** High impressions with 0 clicks trigger the rule. It could be wrong if the impressions are concentrated in low-intent queries.

4. **Rank 4 — REVIEW_PAGE:** High impression volume and 0 clicks produce a high score. It could be wrong if the underlying search demand does not normally lead to clicks.

5. **Rank 5 — REVIEW_PAGE:** High impressions with no clicks make it a priority for manual review. It could be wrong if the data reflects an unusual query mix or incomplete tracking.

6. **Rank 6 — REVIEW_PAGE:** Similar high-volume, zero-click pattern. It could be wrong if the observed impressions do not represent meaningful opportunities.

7. **Rank 7 — REVIEW_PAGE:** High impressions and 0 clicks produce a high score. It could be wrong if the page is visible for queries where clicks are naturally rare.

8. **Rank 8 — REVIEW_PAGE:** High impressions with 0 clicks trigger the rule. It could be wrong because the score does not consider query intent or SERP features.

9. **Rank 9 — REVIEW_PAGE:** Very high impressions but only 1 click gives an extremely low CTR. It could be wrong if impressions come from broad or low-intent searches.

10. **Rank 10 — REVIEW_PAGE:** High impressions with 0 clicks make it a review candidate. It could be wrong if tracking or query composition explains the low clicks.

11. **Rank 11 — REVIEW_PAGE:** High impressions and 0 clicks trigger the rule. It could be wrong if the page is appearing for queries with limited click opportunity.

12. **Rank 12 — REVIEW_PAGE:** High impressions with 0 clicks produce a high score. It could be wrong if the page's impressions are not actionable search opportunities.

13. **Rank 13 — REVIEW_PAGE:** High impressions and 0 clicks trigger the rule. It could be wrong if the observed data is incomplete or affected by query mix.

14. **Rank 14 — REVIEW_PAGE:** High impressions with no clicks make it a priority for review. It could be wrong if the impressions do not correspond to high-intent searches.

15. **Rank 15 — REVIEW_PAGE:** High impressions and 0 clicks produce a high score. It could be wrong if SERP features or query intent reduce click opportunity.

16. **Rank 16 — REVIEW_PAGE:** High impressions with only 1 click create a very low CTR. It could be wrong if the page is receiving impressions from broad, low-intent queries.

17. **Rank 17 — REVIEW_PAGE:** High impressions with 0 clicks trigger the rule. It could be wrong if the page's search visibility does not represent a realistic click opportunity.

18. **Rank 18 — REVIEW_PAGE:** High impressions and 0 clicks produce a high score. It could be wrong if the data is incomplete or the query mix is unusual.

19. **Rank 19 — REVIEW_PAGE:** High impressions with 0 clicks make it a review candidate. It could be wrong if impressions are generated by low-intent searches.

20. **Rank 20 — REVIEW_PAGE:** High impressions and 0 clicks trigger the rule. It could be wrong if the page is visible in SERPs where users have little reason to click.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
review_top20 = queue.nlargest(20, "score")[
    [
        "client_hash_id",
        "content_hash_id",
        "report_date",
        "gsc_impressions",
        "gsc_clicks",
        "ctr",
        "score",
        "action",
        "reason_code"
    ]
].copy()

review_top20.insert(0, "rank", range(1, 21))

display(review_top20)

,rank,client_hash_id,content_hash_id,report_date,gsc_impressions,gsc_clicks,ctr,score,action,reason_code
103661,1,client_62f4a7e64f5e0096,content_945d6ff91386c817,2026-03-04,37368,0,0.000000,37368.0,REVIEW_PAGE,HIGH_VOLUME_LOW_CLICKS
8882581,2,client_73cda7b4e4f265ea,content_fec55986a1868d62,2026-03-30,33383,0,0.000000,33383.0,REVIEW_PAGE,HIGH_VOLUME_LOW_CLICKS
8550024,3,client_23a62021009f63c4,content_44f34c0a90047651,2026-03-27,32958,0,0.000000,32958.0,REVIEW_PAGE,HIGH_VOLUME_LOW_CLICKS
9451315,4,client_73cda7b4e4f265ea,content_fec55986a1868d62,2026-03-31,31472,0,0.000000,31472.0,REVIEW_PAGE,HIGH_VOLUME_LOW_CLICKS
766551,5,client_73cda7b4e4f265ea,content_9c057b66c30a3abb,2026-03-02,28973,0,0.000000,28973.0,REVIEW_PAGE,HIGH_VOLUME_LOW_CLICKS
599344,6,client_73cda7b4e4f265ea,content_9c057b66c30a3abb,2026-03-01,28947,0,0.000000,28947.0,REVIEW_PAGE,HIGH_VOLUME_LOW_CLICKS
6438283,7,client_62f4a7e64f5e0096,content_34a70fea29d15f24,2026-03-22,27410,0,0.000000,27410.0,REVIEW_PAGE,HIGH_VOLUME_LOW_CLICKS
803453,8,client_73cda7b4e4f265ea,content_9c057b66c30a3abb,2026-03-03,24233,0,0.000000,24233.0,REVIEW_PAGE,HIGH_VOLUME_LOW_CLICKS
9655081,9,client_23a62021009f63c4,content_44f34c0a90047651,2026-03-28,40084,1,0.000025,20042.0,REVIEW_PAGE,HIGH_VOLUME_LOW_CLICKS
4062723,10,client_e547b89c05043229,content_757b1fa67827358d,2026-03-13,19301,0,0.000000,19301.0,REVIEW_PAGE,HIGH_VOLUME_LOW_CLICKS


## 4. Weak picks + leakage check

The weakest part of this baseline is that the score uses only impressions and clicks.
It does not account for query intent, SERP features, page type, or other contextual
factors. Therefore, high-scoring rows should be treated as review candidates rather
than confirmed problems.

No future-window labels or product decision outputs were used in the baseline score.
The rule uses only March 2026 observed GSC impressions and clicks.

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 4 — Weak picks + leakage check

print("Top 5 priority picks:")
display(
    review_top20.tail(5)[
        [
            "rank",
            "gsc_impressions",
            "gsc_clicks",
            "ctr",
            "score",
            "action",
            "reason_code"
        ]
    ]
)

# Check for leakage fields
leakage_fields = [
    "trend_direction",
    "health_score",
    "priority_score",
    "action_type"
]

print("\nLeakage fields present in source data:")

for field in leakage_fields:
    print(f"{field}: {field in df_march.columns}")

print("\nFeatures used in baseline score:")
print(["gsc_impressions", "gsc_clicks"])

print("\nNo future-window labels or existing decision outputs were used.")

Top 5 priority picks:


,rank,gsc_impressions,gsc_clicks,ctr,score,action,reason_code
7822469,16,30964,1,0.000032,15482.0,REVIEW_PAGE,HIGH_VOLUME_LOW_CLICKS
103514,17,13827,0,0.000000,13827.0,REVIEW_PAGE,HIGH_VOLUME_LOW_CLICKS
8365888,18,13788,0,0.000000,13788.0,REVIEW_PAGE,HIGH_VOLUME_LOW_CLICKS
8095571,19,13764,0,0.000000,13764.0,REVIEW_PAGE,HIGH_VOLUME_LOW_CLICKS
879923,20,13726,0,0.000000,13726.0,REVIEW_PAGE,HIGH_VOLUME_LOW_CLICKS



Leakage fields present in source data:
trend_direction: False
health_score: False
priority_score: False
action_type: False

Features used in baseline score:
['gsc_impressions', 'gsc_clicks']

No future-window labels or existing decision outputs were used.


## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.